In [ ]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz
import joblib

item_embeddings = np.load('data/processed/item_embeddings.npy')
train_fe = pd.read_parquet('data/processed/train_fe.parquet')
queries_fe = pd.read_parquet('data/processed/queries_fe.parquet')
items_fe = pd.read_parquet('data/processed/items_fe.parquet')
item_tfidf_matrix = load_npz('data/processed/item_tfidf_matrix.npz')
tfidf_vectorizer = joblib.load('data/processed/tfidf_vectorizer.pkl')

print("Установка и настройка BM25\n")

nltk.download('punkt', quiet=True)

print("BM25 готов к использованию")

Установка и настройка BM25

BM25 готов к использованию


In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

print("Установка и настройка NLTK для токенизации\n")

print("Скачивание токенизаторов NLTK...")
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("NLTK готов к использованию")

# Создание BM25 индекса с учетом русского языка
print("\nСоздание BM25 индекса\n")

# Тексты объявлений
item_texts = items_fe['item_text_combined'].fillna('').tolist()

print(f"Токенизация {len(item_texts)} объявлений...")

tokenized_items = [word_tokenize(str(text).lower(), language='russian') for text in item_texts]

# Создаем BM25 индекс
print("Создание BM25 индекса...")
bm25_index = BM25Okapi(tokenized_items)

print(f"\nBM25 индекс создан:")
print(f"   Количество документов: {bm25_index.corpus_size}")

avg_tokens = sum(len(t) for t in tokenized_items) / len(tokenized_items)
print(f"   Среднее количество токенов на документ: {avg_tokens:.1f}")

Установка и настройка NLTK для токенизации

Скачивание токенизаторов NLTK...
NLTK готов к использованию

Создание BM25 индекса

Токенизация 189212 объявлений...
Создание BM25 индекса...

BM25 индекс создан:
   Количество документов: 189212
   Среднее количество токенов на документ: 188.2


In [10]:
def find_candidates_bm25(query_text, bm25_index, items_df, top_n=50):
    # Токенизируем запрос с учетом русского языка
    tokenized_query = word_tokenize(str(query_text).lower(), language='russian')
    
    # Получаем BM25 скоры
    bm25_scores = bm25_index.get_scores(tokenized_query)
    
    # Находим топ-N индексов
    top_indices = np.argsort(bm25_scores)[::-1][:top_n]
    
    # Получаем item_id
    top_item_ids = items_df.iloc[top_indices]['item_id'].tolist()
    
    return top_item_ids

# Тест BM25
print("Тест BM25 поиска...")
test_query = queries_fe.iloc[0]['search_query_clean']
print(f"Запрос: '{test_query}'")

candidates_bm25 = find_candidates_bm25(
    test_query, 
    bm25_index, 
    items_fe, 
    top_n=10
)

print(f"\nНайдено {len(candidates_bm25)} кандидатов:")
for i, item_id in enumerate(candidates_bm25[:5], 1):
    item = items_fe[items_fe['item_id'] == item_id].iloc[0]
    print(f"  {i}. {item['item_title_raw'][:80]}")

Тест BM25 поиска...
Запрос: 'перевозки владикавказ тбилиси'

Найдено 10 кандидатов:
  1. Трансфер в Грузию и Обратно
  2. Трансфер из России в Грузию Владикавказ - Батуми
  3. Трансфер в Тбилиси,Гудаури,поездки в Грузию
  4. Трансфер Владикавказ Тбилиси без предоплаты
  5. Сочи Краснодар Владикавказ Тбилиси Ереван


In [12]:
import faiss
import numpy as np

print("Создание FAISS индекса\n")

# Используем существующие эмбеддинги
item_embeddings = np.load('data/processed/item_embeddings.npy')

print(f"Загрузка эмбеддингов: {item_embeddings.shape}")

# Создаем FAISS индекс
embedding_dim = item_embeddings.shape[1]
print(f"Размерность эмбеддингов: {embedding_dim}")

# Нормализуем эмбеддинги для косинусного сходства
faiss.normalize_L2(item_embeddings)

# Создаем индекс
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(item_embeddings)

print(f"\nFAISS индекс создан:")
print(f"   Тип: IndexFlatIP")
print(f"   Количество векторов: {faiss_index.ntotal}")
print(f"   Размерность: {faiss_index.d}")

Создание FAISS индекса

Загрузка эмбеддингов: (189212, 768)
Размерность эмбеддингов: 768

FAISS индекс создан:
   Тип: IndexFlatIP
   Количество векторов: 189212
   Размерность: 768


In [13]:
from sentence_transformers import SentenceTransformer
import torch

print("Загрузка Sentence Transformer модели\n")

# Проверяем доступность GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Используемое устройство: {device}")

# Загрузка модели multilingual
model_name = 'paraphrase-multilingual-mpnet-base-v2'
print(f"Загрузка модели: {model_name}")

model = SentenceTransformer(model_name, device=device)

print(f"Модель загружена")
print(f"   Размерность эмбеддингов: {model.get_embedding_dimension()}")

Загрузка Sentence Transformer модели

Используемое устройство: cpu
Загрузка модели: paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Модель загружена
   Размерность эмбеддингов: 768


In [15]:
def find_candidates_faiss(query_text, model, faiss_index, items_df, top_n=50):
    # Получаем эмбеддинг запроса
    query_embedding = model.encode([query_text], convert_to_numpy=True)
    
    # Нормализуем
    faiss.normalize_L2(query_embedding)
    
    # Ищем ближайших соседей
    distances, indices = faiss_index.search(query_embedding, top_n)
    
    # Получаем ID объявлений
    top_item_ids = items_df.iloc[indices[0]]['item_id'].tolist()
    
    return top_item_ids

# Тест FAISS
print("Тест FAISS поиска...")
test_query = queries_fe.iloc[0]['search_query_clean']
print(f"Запрос: '{test_query}'")

candidates_faiss = find_candidates_faiss(
    test_query, 
    model, 
    faiss_index, 
    items_fe, 
    top_n=10
)

print(f"\nНайдено {len(candidates_faiss)} кандидатов:")
for i, item_id in enumerate(candidates_faiss[:5], start=1):
    item = items_fe[items_fe['item_id'] == item_id].iloc[0]
    print(f"  {i}. {item['item_title_raw'][:80]}")

Тест FAISS поиска...
Запрос: 'перевозки владикавказ тбилиси'

Найдено 10 кандидатов:
  1. Услуги перевозки людей
  2. Трансфер минивэн из Краснодара в Тбилиси и Ереван
  3. Трансфер из Нальчика в Грузию и Армению
  4. Трансфер Владикавказ - Тбилиси, Батуми, Ереван
  5. Услуги Перевозки Машины из Армении Грузии в Россию


In [ ]:
def find_candidates_ensemble_extended(query_row, tfidf_vectorizer, item_tfidf_matrix,
                                     bm25_index, model, faiss_index, items_df,
                                     top_n=50, tfidf_weight=0.2, bm25_weight=0.35, emb_weight=0.45,
                                     location_boost=True):

    query_text = query_row['search_query_clean']
    query_location = query_row.get('search_location_filled', None)
    
    # TF-IDF сходство
    query_tfidf = tfidf_vectorizer.transform([query_text])
    tfidf_similarities = cosine_similarity(query_tfidf, item_tfidf_matrix).flatten()
    
    # BM25 сходство
    tokenized_query = word_tokenize(query_text.lower())
    bm25_scores = bm25_index.get_scores(tokenized_query)
    
    # FAISS эмбеддинги
    query_embedding = model.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    distances, indices = faiss_index.search(query_embedding, len(items_df))
    emb_similarities = np.zeros(len(items_df))
    for idx, dist in zip(indices[0], distances[0]):
        if idx < len(items_df):
            emb_similarities[idx] = dist
    
    # Нормализуем все сходства
    tfidf_norm = (tfidf_similarities - tfidf_similarities.min()) / (tfidf_similarities.max() - tfidf_similarities.min() + 1e-8)
    bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-8)
    emb_norm = (emb_similarities - emb_similarities.min()) / (emb_similarities.max() - emb_similarities.min() + 1e-8)
    
    # Комбинируем
    combined_similarities = (tfidf_weight * tfidf_norm + 
                            bm25_weight * bm25_norm + 
                            emb_weight * emb_norm)
    
    # Буст по локации
    if query_location and query_location != -1 and location_boost:
        same_location_mask = items_df['item_location_filled'] == query_location
        if same_location_mask.sum() > 0:
            combined_similarities[same_location_mask.values] *= 1.5
    
    # Топ-N
    top_indices = np.argsort(combined_similarities)[::-1][:top_n]
    top_item_ids = items_df.iloc[top_indices]['item_id'].tolist()
    
    return top_item_ids

# Тест расширенного ансамбля
print("Тест расширенного ансамбля...")
test_query_row = queries_fe.iloc[0]
print(f"Запрос: '{test_query_row['search_query_clean']}'")

candidates_ext = find_candidates_ensemble_extended(
    test_query_row, 
    tfidf_vectorizer, 
    item_tfidf_matrix, 
    bm25_index, 
    model, 
    faiss_index, 
    items_fe, 
    top_n=10,
    tfidf_weight=0.2,
    bm25_weight=0.35,
    emb_weight=0.45,
    location_boost=True
)

print(f"\nНайдено {len(candidates_ext)} кандидатов:")
for i, item_id in enumerate(candidates_ext[:5], start=1):
    item = items_fe[items_fe['item_id'] == item_id].iloc[0]
    print(f"  {i}. {item['item_title_raw'][:80]}")

Тест расширенного ансамбля...
Запрос: 'перевозки владикавказ тбилиси'

Найдено 10 кандидатов:
  1. Трансфер в Тбилиси,Гудаури,поездки в Грузию
  2. Трансфер в Грузию и Обратно
  3. Трансфер из России в Грузию Владикавказ - Батуми
  4. Трансфер Владикавказ - Тбилиси.Батуми Гудаури
  5. Сочи Краснодар Владикавказ Тбилиси Ереван


In [19]:
import time

print("ГЕНЕРАЦИЯ ФИНАЛЬНЫХ ПРЕДСКАЗАНИЙ")

# Используем лучшие веса из подбора
BEST_TFIDF_WEIGHT = 0.2
BEST_BM25_WEIGHT = 0.35
BEST_EMB_WEIGHT = 0.45

print(f"\nПараметры:")
print(f"  TF-IDF вес: {BEST_TFIDF_WEIGHT}")
print(f"  BM25 вес: {BEST_BM25_WEIGHT}")
print(f"  Emb вес: {BEST_EMB_WEIGHT}")
print(f"  Location boost: True")
print(f"  Топ-N: 50")

# Словарь для хранения предсказаний
final_predictions = {}

# Всего запросов
total_queries = len(queries_fe)
print(f"\nВсего запросов для обработки: {total_queries}")

# Засекаем время
start_time = time.time()

print(f"\nНачинаем генерацию предсказаний...")
print("-" * 60)

for idx, row in queries_fe.iterrows():
    query_id = row['query_id']
    
    try:
        # Находим кандидатов
        candidates = find_candidates_ensemble_extended(
            row,
            tfidf_vectorizer,
            item_tfidf_matrix,
            bm25_index,
            model,
            faiss_index,
            items_fe,
            top_n=50,
            tfidf_weight=BEST_TFIDF_WEIGHT,
            bm25_weight=BEST_BM25_WEIGHT,
            emb_weight=BEST_EMB_WEIGHT,
            location_boost=True
        )
        
        # Сохраняем предсказания
        final_predictions[query_id] = candidates
        
    except Exception as e:
        print(f"  Ошибка для запроса {query_id}: {e}")
        # В случае ошибки сохраняем пустой список
        final_predictions[query_id] = []
    
    # Показываем прогресс каждые 100 запросов
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        avg_time_per_query = elapsed / (idx + 1)
        remaining_time = avg_time_per_query * (total_queries - idx - 1)
        
        print(f"  Прогресс: {idx + 1}/{total_queries} ({(idx+1)/total_queries*100:.1f}%)")

# Финальное время
total_time = time.time() - start_time
print(f"\n Генерация завершена")
print(f"  Обработано запросов: {len(final_predictions)}")
print(f"  Общее время: {total_time:.1f} сек ({total_time/60:.1f} мин)")
print(f"  Среднее время на запрос: {total_time/len(final_predictions)*1000:.1f} мс")

ГЕНЕРАЦИЯ ФИНАЛЬНЫХ ПРЕДСКАЗАНИЙ

Параметры:
  TF-IDF вес: 0.2
  BM25 вес: 0.35
  Emb вес: 0.45
  Location boost: True
  Топ-N: 50

Всего запросов для обработки: 2452

Начинаем генерацию предсказаний...
------------------------------------------------------------
  Прогресс: 100/2452 (4.1%)
  Прогресс: 200/2452 (8.2%)
  Прогресс: 300/2452 (12.2%)
  Прогресс: 400/2452 (16.3%)
  Прогресс: 500/2452 (20.4%)
  Прогресс: 600/2452 (24.5%)
  Прогресс: 700/2452 (28.5%)
  Прогресс: 800/2452 (32.6%)
  Прогресс: 900/2452 (36.7%)
  Прогресс: 1000/2452 (40.8%)
  Прогресс: 1100/2452 (44.9%)
  Прогресс: 1200/2452 (48.9%)
  Прогресс: 1300/2452 (53.0%)
  Прогресс: 1400/2452 (57.1%)
  Прогресс: 1500/2452 (61.2%)
  Прогресс: 1600/2452 (65.3%)
  Прогресс: 1700/2452 (69.3%)
  Прогресс: 1800/2452 (73.4%)
  Прогресс: 1900/2452 (77.5%)
  Прогресс: 2000/2452 (81.6%)
  Прогресс: 2100/2452 (85.6%)
  Прогресс: 2200/2452 (89.7%)
  Прогресс: 2300/2452 (93.8%)
  Прогресс: 2400/2452 (97.9%)

 Генерация завершена
  Обр

In [24]:
import pickle


# Сохраняем в pickle
with open('data/processed/final_predictions.pkl', 'wb') as f:
    pickle.dump(final_predictions, f)
print(f"Сохранено в final_predictions.pkl")


# Создаем DataFrame для удобной работы
predictions_df = pd.DataFrame({
    'query_id': list(final_predictions.keys()),
    'candidates': list(final_predictions.values())
})

print(f"\nРазмер данных:")
print(f"  {len(predictions_df)} строк")

print("\nВсе предсказания сохранены")

Сохранено в final_predictions.pkl

Размер данных:
  2452 строк

Все предсказания сохранены
